<a href="https://colab.research.google.com/github/bshinib483/Img-detection/blob/main/Train2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install torch torchvision ultralytics pandas pillow matplotlib tqdm scikit-learn kaggle opencv-python


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.4 MB/s eta 0:00:00


In [2]:
import os, json

kaggle_dir = os.path.expanduser('~/.kaggle')
os.makedirs(kaggle_dir, exist_ok=True)

kaggle_json = {
    'username': 'shruti483',
    'key': 'KGAT_e18981c94b1d5eb7616a3cd9ad2683c8'
}

with open(f'{kaggle_dir}/kaggle.json', 'w') as f:
    json.dump(kaggle_json, f)

os.chmod(f'{kaggle_dir}/kaggle.json', 0o600)
print('kaggle.json created')


kaggle.json created


In [3]:
!kaggle datasets download -d a2015003713/militaryaircraftdetectiondataset
import zipfile

with zipfile.ZipFile(
    "militaryaircraftdetectiondataset.zip",
    'r'
) as zip_ref:

    zip_ref.extractall("dataset_files")

print("Dataset extracted")

Dataset URL: https://www.kaggle.com/datasets/a2015003713/militaryaircraftdetectiondataset
License(s): unknown
100% 11.2G/11.2G [10:30<00:00, 19.0MB/s]

Dataset extracted


In [4]:
import os

print(os.listdir("dataset_files"))

['labels_with_split.csv', 'annotated_samples', 'dataset', 'crop']


In [5]:
import pandas as pd

df = pd.read_csv(
    "dataset_files/labels_with_split.csv"
)

df.head()


,filename,width,height,class,xmin,ymin,xmax,ymax,split
0,0000e97ea2d086d6759b19b288a8a72c,4928,3264,Mi28,1380,1904,1522,2014,train
1,0000e97ea2d086d6759b19b288a8a72c,4928,3264,Mi28,1809,1625,1958,1759,train
2,0000e97ea2d086d6759b19b288a8a72c,4928,3264,Mi28,2400,1571,2532,1727,train
3,0000e97ea2d086d6759b19b288a8a72c,4928,3264,Mi28,3935,1772,4100,1891,train
4,00010041af654d0b8e1e16c824fa9867,1360,2048,UH60,835,526,1233,741,train


In [6]:
import os
import shutil
from tqdm import tqdm

# OUTPUT DIRECTORY
output_dir = "aircraft_yolo"

# IMAGE ROOT
image_src = "dataset_files/dataset"


selected_classes = [
    "Rafale",
    "Su57",
    "F22",
    "F35",
    "Tejas",
    "Mig29",
    "Mirage2000",
    "C17",
    "C130",
    "CH47",
    "AH64"
]

# FILTER DATAFRAME
df = df[df["class"].isin(selected_classes)]

# CLASSES
classes = sorted(df["class"].unique())

# CLASS IDS
class_to_id = {
    cls: idx for idx, cls in enumerate(classes)
}

print(class_to_id)

# FIND ALL JPG IMAGES
all_images = {}

for root, dirs, files in os.walk(image_src):

    for file in files:

        if file.endswith(".jpg"):

            all_images[file] = os.path.join(root, file)

print("Total images found:", len(all_images))

# CREATE YOLO DATASET
for _, row in tqdm(df.iterrows(), total=len(df)):

    # IMPORTANT FIX
    filename = row["filename"] + ".jpg"

    label = row["class"]

    split = row["split"]

    width = row["width"]
    height = row["height"]

    xmin = row["xmin"]
    ymin = row["ymin"]
    xmax = row["xmax"]
    ymax = row["ymax"]

    # YOLO FORMAT
    x_center = ((xmin + xmax) / 2) / width

    y_center = ((ymin + ymax) / 2) / height

    bbox_width = (xmax - xmin) / width

    bbox_height = (ymax - ymin) / height

    # CREATE FOLDERS
    image_folder = os.path.join(
        output_dir,
        "images",
        split
    )

    label_folder = os.path.join(
        output_dir,
        "labels",
        split
    )

    os.makedirs(image_folder, exist_ok=True)

    os.makedirs(label_folder, exist_ok=True)

    # COPY IMAGE
    if filename in all_images:

        src_img = all_images[filename]

        dst_img = os.path.join(
            image_folder,
            filename
        )

        shutil.copy(src_img, dst_img)

        # CREATE LABEL
        txt_name = filename.replace(
            ".jpg",
            ".txt"
        )

        txt_path = os.path.join(
            label_folder,
            txt_name
        )

        class_id = class_to_id[label]

        with open(txt_path, "w") as f:

            f.write(
                f"{class_id} "
                f"{x_center} "
                f"{y_center} "
                f"{bbox_width} "
                f"{bbox_height}"
            )

print("YOLO dataset ready")

{'AH64': 0, 'C130': 1, 'C17': 2, 'CH47': 3, 'F22': 4, 'F35': 5, 'Mig29': 6, 'Mirage2000': 7, 'Rafale': 8, 'Su57': 9, 'Tejas': 10}
Total images found: 23143


100%|██████████| 7903/7903 [00:35<00:00, 225.14it/s]

YOLO dataset ready


In [7]:
print(os.listdir("aircraft_yolo/images"))

print(
    "Train:",
    len(os.listdir("aircraft_yolo/images/train"))
)

print(
    "Validation:",
    len(os.listdir("aircraft_yolo/images/validation"))
)

print(
    "Test:",
    len(os.listdir("aircraft_yolo/images/test"))
)

['validation', 'test', 'train']
Train: 3327
Validation: 857
Test: 292


In [8]:
yaml_text = f"""
path: aircraft_yolo

train: images/train
val: images/validation
test: images/test

names:
"""

for idx, cls in enumerate(classes):

    yaml_text += f"  {idx}: {cls}\n"

with open("aircraft.yaml", "w") as f:

    f.write(yaml_text)

print("aircraft.yaml created")

aircraft.yaml created


In [10]:
from ultralytics import YOLO

# LOAD YOLOv8
model = YOLO("yolov8n.pt")

# TRAIN
model.train(
    data="aircraft.yaml",
    epochs=48,
    imgsz=640,
    batch=16
)

Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=aircraft.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=48, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train-2, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=100, perspective=0

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7d5eb60f73b0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.

In [13]:
import os

print(df.iloc[0])

label = df.iloc[0]['class']

filename = str(df.iloc[0]['filename'])

print("\nLABEL:", label)

print("FILENAME:", filename)

print("\nChecking paths:\n")

print(
    os.path.join(
        "dataset_files/crop",
        label
    )
)

print(
    os.listdir(
        os.path.join(
            "dataset_files/crop",
            label
        )
    )[:5]
)

filename    0003f56298fa8999168d7988a2e9549d
width                                   4046
height                                  2697
class                                    F22
xmin                                     213
ymin                                     505
xmax                                    3550
ymax                                    1623
split                                  train
Name: 23, dtype: object

LABEL: F22
FILENAME: 0003f56298fa8999168d7988a2e9549d

Checking paths:

dataset_files/crop/F22
['fccdada1b7cfd498501c397ba9cc92d9_2.jpg', '8d13af9fddccacd38ae228a90ae5dde4_0.jpg', '427507420a109dc30caa4971f76b0aa2_0.jpg', '88e9cc677398c451f3a8e14f35b5db2e_1.jpg', '9aae9b74843dd2682481cf6771cbfea7_0.jpg']


In [14]:
import os
import shutil
import random
from tqdm import tqdm

random.seed(42)

source_dir = "dataset_files/crop"

output_dir = "aircraft_classifier"

# DELETE OLD
shutil.rmtree(
    output_dir,
    ignore_errors=True
)

classes = [
    "Rafale",
    "Su57",
    "F22",
    "F35",
    "Tejas",
    "Mig29",
    "Mirage2000",
    "C17",
    "C130",
    "CH47",
    "AH64"
]

for cls in classes:

    class_path = os.path.join(
        source_dir,
        cls
    )

    images = os.listdir(class_path)

    random.shuffle(images)

    total = len(images)

    train_split = int(0.8 * total)

    valid_split = int(0.1 * total)

    train_images = images[:train_split]

    valid_images = images[
        train_split:
        train_split + valid_split
    ]

    test_images = images[
        train_split + valid_split:
    ]

    splits = {

        "train": train_images,

        "validation": valid_images,

        "test": test_images
    }

    for split_name, split_images in splits.items():

        split_folder = os.path.join(

            output_dir,

            split_name,

            cls
        )

        os.makedirs(
            split_folder,
            exist_ok=True
        )

        for img_name in tqdm(split_images):

            src = os.path.join(
                class_path,
                img_name
            )

            dst = os.path.join(
                split_folder,
                img_name
            )

            shutil.copy(src, dst)

print("Classifier dataset ready")

100%|██████████| 58/58 [00:00<00:00, 630.89it/s]

Classifier dataset ready


In [16]:
import os

print(
    os.listdir(
        "aircraft_classifier/train"
    )
)

print(
    len(os.listdir(
        "aircraft_classifier/train/F35"
    ))
)

print(
    len(os.listdir(
        "aircraft_classifier/validation/F35"
    ))
)

['C130', 'C17', 'Mig29', 'Mirage2000', 'CH47', 'F22', 'F35', 'Su57', 'Rafale', 'Tejas', 'AH64']
1255
156


In [17]:
import torch

from torchvision import datasets
from torchvision import transforms

from torch.utils.data import DataLoader

transform = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(10),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2
    ),

    transforms.ToTensor()
])

train_dataset = datasets.ImageFolder(

    "aircraft_classifier/train",

    transform=transform
)

valid_dataset = datasets.ImageFolder(

    "aircraft_classifier/validation",

    transform=transform
)

train_loader = DataLoader(

    train_dataset,

    batch_size=32,

    shuffle=True
)

valid_loader = DataLoader(

    valid_dataset,

    batch_size=32
)

print(train_dataset.classes)

['AH64', 'C130', 'C17', 'CH47', 'F22', 'F35', 'Mig29', 'Mirage2000', 'Rafale', 'Su57', 'Tejas']


In [18]:
import torch.nn as nn

from torchvision import models

device = torch.device(

    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

model_cls = models.efficientnet_b0(
    pretrained=True
)

num_classes = len(
    train_dataset.classes
)

model_cls.classifier[1] = nn.Linear(

    model_cls.classifier[1].in_features,

    num_classes
)

model_cls = model_cls.to(device)

print(model_cls)

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 188MB/s]

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [19]:
import torch.optim as optim

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(

    model_cls.parameters(),

    lr=0.0003
)

In [21]:
epochs = 20

best_accuracy = 0

for epoch in range(epochs):

    # =========================
    # TRAINING
    # =========================

    model_cls.train()

    running_loss = 0

    train_correct = 0

    train_total = 0

    for images, labels in train_loader:

        images = images.to(device)

        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model_cls(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        train_total += labels.size(0)

        train_correct += (
            predicted == labels
        ).sum().item()

    train_accuracy = (
        100 * train_correct / train_total
    )

    # =========================
    # VALIDATION
    # =========================

    model_cls.eval()

    valid_correct = 0

    valid_total = 0

    valid_loss = 0

    with torch.no_grad():

        for images, labels in valid_loader:

            images = images.to(device)

            labels = labels.to(device)

            outputs = model_cls(images)

            loss = criterion(outputs, labels)

            valid_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            valid_total += labels.size(0)

            valid_correct += (
                predicted == labels
            ).sum().item()

    valid_accuracy = (
        100 * valid_correct / valid_total
    )

    # =========================
    # SAVE BEST MODEL
    # =========================

    if valid_accuracy > best_accuracy:

        best_accuracy = valid_accuracy

        torch.save(

            model_cls.state_dict(),

            "best_aircraft_classifier.pth"
        )

        print("BEST MODEL SAVED")

    # =========================
    # PRINT STATS
    # =========================

    print("\n=========================")

    print(f"Epoch {epoch+1}/{epochs}")

    print("=========================")

    print(f"Train Loss: {running_loss:.4f}")

    print(f"Train Accuracy: {train_accuracy:.2f}%")

    print(f"Validation Loss: {valid_loss:.4f}")

    print(f"Validation Accuracy: {valid_accuracy:.2f}%")

    print(f"Best Validation Accuracy: {best_accuracy:.2f}%")

    # =========================
    # OVERFITTING CHECK
    # =========================

    gap = train_accuracy - valid_accuracy

    print(f"Accuracy Gap: {gap:.2f}%")

    if gap > 10:

        print("WARNING: Possible Overfitting")

BEST MODEL SAVED

Epoch 1/20
Train Loss: 126.1306
Train Accuracy: 79.50%
Validation Loss: 10.6920
Validation Accuracy: 86.11%
Best Validation Accuracy: 86.11%
Accuracy Gap: -6.61%
BEST MODEL SAVED

Epoch 2/20
Train Loss: 59.5265
Train Accuracy: 90.68%
Validation Loss: 8.8291
Validation Accuracy: 88.15%
Best Validation Accuracy: 88.15%
Accuracy Gap: 2.52%
BEST MODEL SAVED

Epoch 3/20
Train Loss: 35.1585
Train Accuracy: 94.37%
Validation Loss: 8.3065
Validation Accuracy: 90.19%
Best Validation Accuracy: 90.19%
Accuracy Gap: 4.17%
BEST MODEL SAVED

Epoch 4/20
Train Loss: 25.7857
Train Accuracy: 95.85%
Validation Loss: 7.8881
Validation Accuracy: 90.57%
Best Validation Accuracy: 90.57%
Accuracy Gap: 5.28%
BEST MODEL SAVED

Epoch 5/20
Train Loss: 20.0591
Train Accuracy: 96.74%
Validation Loss: 7.1260
Validation Accuracy: 91.46%
Best Validation Accuracy: 91.46%
Accuracy Gap: 5.27%
BEST MODEL SAVED

Epoch 6/20
Train Loss: 15.8737
Train Accuracy: 97.50%
Validation Loss: 7.4391
Validation Accur

In [22]:
!pip install scikit-learn

In [23]:
import torch
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

# =========================
# EVALUATION MODE
# =========================

model_cls.eval()

all_labels = []

all_predictions = []

# =========================
# NO GRADIENTS
# =========================

with torch.no_grad():

    for images, labels in valid_loader:

        images = images.to(device)

        labels = labels.to(device)

        outputs = model_cls(images)

        _, predicted = torch.max(
            outputs,
            1
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

        all_predictions.extend(
            predicted.cpu().numpy()
        )

# =========================
# METRICS
# =========================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    average="weighted"
)

recall = recall_score(
    all_labels,
    all_predictions,
    average="weighted"
)

f1 = f1_score(
    all_labels,
    all_predictions,
    average="weighted"
)

# =========================
# PRINT RESULTS
# =========================

print("\n========================")
print("MODEL EVALUATION")
print("========================")

print(f"\nAccuracy  : {accuracy:.4f}")

print(f"Precision : {precision:.4f}")

print(f"Recall    : {recall:.4f}")

print(f"F1 Score  : {f1:.4f}")

# =========================
# CONFUSION MATRIX
# =========================

cm = confusion_matrix(
    all_labels,
    all_predictions
)

print("\nConfusion Matrix:\n")

print(cm)

# =========================
# CLASSIFICATION REPORT
# =========================

print("\nClassification Report:\n")

print(

    classification_report(

        all_labels,

        all_predictions,

        target_names=train_dataset.classes
    )
)


MODEL EVALUATION

Accuracy  : 0.9223
Precision : 0.9245
Recall    : 0.9223
F1 Score  : 0.9217

Confusion Matrix:

[[ 54   0   0   1   0   1   0   0   0   0   0]
 [  0 154   2   1   0   1   0   1   0   0   0]
 [  0   3  67   1   0   0   0   0   2   1   0]
 [  1   0   0  37   0   0   0   0   0   0   0]
 [  0   1   0   1  65   3   0   1   0   0   0]
 [  0   3   0   2   4 143   0   0   3   1   0]
 [  0   2   0   0   3   2  29   1   1   1   1]
 [  0   0   0   0   0   0   0  44   2   0   0]
 [  0   2   0   0   1   0   1   0  76   1   1]
 [  0   2   0   0   2   0   0   0   0  43   0]
 [  0   1   0   1   0   0   0   0   2   0  12]]

Classification Report:

              precision    recall  f1-score   support

        AH64       0.98      0.96      0.97        56
        C130       0.92      0.97      0.94       159
         C17       0.97      0.91      0.94        74
        CH47       0.84      0.97      0.90        38
         F22       0.87      0.92      0.89        71
         F35     

In [24]:
torch.save(

    model_cls.state_dict(),

    "efficientnet_aircraft.pth"
)

print("Classifier saved")

Classifier saved


In [49]:
from google.colab import files

uploaded = files.upload()

Saving 2.jpg to 2.jpg


In [50]:
from ultralytics import YOLO

detector = YOLO(
    "runs/detect/train/weights/best.pt"
)

results = detector(
    "2.jpg"
)

results[0].save("detected.jpg")

print(results)


image 1/1 /content/2.jpg: 416x640 (no detections), 8.5ms
Speed: 2.3ms preprocess, 8.5ms inference, 0.6ms postprocess per image at shape (1, 3, 416, 640)
[ultralytics.engine.results.Results object with attributes:

boxes: ultralytics.engine.results.Boxes object
keypoints: None
masks: None
names: {0: 'AH64', 1: 'C130', 2: 'C17', 3: 'CH47', 4: 'F22', 5: 'F35', 6: 'Mig29', 7: 'Mirage2000', 8: 'Rafale', 9: 'Su57', 10: 'Tejas'}
obb: None
orig_img: array([[[191, 186, 185],
        [191, 186, 185],
        [191, 186, 185],
        ...,
        [183, 178, 177],
        [183, 178, 177],
        [183, 178, 177]],

       [[191, 186, 185],
        [191, 186, 185],
        [191, 186, 185],
        ...,
        [183, 178, 177],
        [183, 178, 177],
        [183, 178, 177]],

       [[191, 186, 185],
        [191, 186, 185],
        [191, 186, 185],
        ...,
        [183, 178, 177],
        [183, 178, 177],
        [183, 178, 177]],

       ...,

       [[211, 217, 222],
        [212, 218, 2

In [56]:
import cv2

image = cv2.imread("2.jpg")

results = detector("2.jpg")

boxes = results[0].boxes.xyxy.cpu().numpy()

# CHECK DETECTIONS
if len(boxes) == 0:

    print("No aircraft detected")

else:

    x1, y1, x2, y2 = boxes[0]

    x1 = int(x1)
    y1 = int(y1)
    x2 = int(x2)
    y2 = int(y2)

    crop = image[y1:y2, x1:x2]

    cv2.imwrite(
        "cropped_aircraft.jpg",
        crop
    )

    print("Aircraft cropped")


image 1/1 /content/2.jpg: 416x640 (no detections), 14.7ms
Speed: 2.9ms preprocess, 14.7ms inference, 1.1ms postprocess per image at shape (1, 3, 416, 640)
No aircraft detected


In [57]:
import torch

from torchvision import models

import torch.nn as nn

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

class_names = train_dataset.classes

model_cls = models.efficientnet_b0(
    pretrained=False
)

model_cls.classifier[1] = nn.Linear(
    model_cls.classifier[1].in_features,
    len(class_names)
)

model_cls.load_state_dict(
    torch.load(
        "efficientnet_aircraft.pth",
        map_location=device
    )
)

model_cls = model_cls.to(device)

model_cls.eval()

EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          (2): Conv2dNormActivat

In [58]:
from PIL import Image

transform_test = transforms.Compose([

    transforms.Resize((224,224)),

    transforms.ToTensor()
])

img = Image.open(
    "cropped_aircraft.jpg"
).convert("RGB")

img = transform_test(img)

img = img.unsqueeze(0).to(device)

with torch.no_grad():

    outputs = model_cls(img)

    _, pred = torch.max(outputs, 1)

predicted_class = class_names[
    pred.item()
]

print(
    "Predicted Aircraft:",
    predicted_class
)

Predicted Aircraft: Su57


In [59]:
torch.save(
    model_cls.state_dict(),
    "aircraft_classifier_efficientnet.pth"
)

print("Model saved successfully")

Model saved successfully


In [39]:
from google.colab import files

files.download(
    "/content/aircraft_classifier_efficientnet.pth"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [40]:
files.download(
    "/content/runs/detect/train/weights/best.pt"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
import json

class_names = train_dataset.classes

with open("class_names.json", "w") as f:

    json.dump(class_names, f)

print("class_names.json saved")

class_names.json saved


In [42]:
files.download(
    "/content/class_names.json"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>